操作系统和计算机网络是后端面试里最容易“看起来很基础，但一追问就露馅”的部分。

如果只是背：

进程和线程的区别。
TCP 三次握手。
HTTP 和 HTTPS 区别。
select、poll、epoll 区别。
top、ps、netstat 常用命令。

这远远不够。

Go 后端面试里，这一章最好和 Go runtime、net/http、goroutine、context、pprof、Linux 排障结合起来回答。

本章可以分成两大部分。

第一部分是操作系统：

1. 进程、线程、协程、goroutine 的关系
2. 用户态和内核态
3. 系统调用、中断、异常
4. 上下文切换为什么贵
5. 虚拟内存、分页、页表、TLB、缺页中断
6. 堆、栈、mmap、Page Cache
7. 死锁条件和避免方式
8. IO 模型、select、poll、epoll、Reactor、零拷贝
9. CPU 缓存、局部性、缓存行、伪共享、MESI
10. Linux 常用排障命令
11. Go 服务线上排障：pprof、trace、goroutine dump、连接池、日志

第二部分是计算机网络：

1. 网络为什么分层
2. OSI 七层和 TCP/IP 四层
3. MAC、IP、端口、Socket
4. TCP 和 UDP 区别
5. TCP 三次握手、四次挥手
6. TIME_WAIT、CLOSE_WAIT、SYN Flood
7. 滑动窗口、流量控制、拥塞控制
8. TCP 粘包拆包和 Go 长连接协议设计
9. HTTP/1.1、HTTP/2、HTTP/3
10. HTTPS 和 TLS 握手
11. DNS、CDN、输入 URL 到页面展示
12. Go HTTP client/server 工程实践
13. 网络排查：ping、traceroute、dig、curl、ss、tcpdump
14. Go 项目里超时、重试、连接复用、幂等、熔断的关系

面试里可以先用一句话总起：

操作系统决定了 Go 服务如何使用 CPU、内存、线程、文件和网络；计算机网络决定了 Go 服务如何通过 TCP/HTTP/RPC 和其他系统通信。Go 后端不需要直接写 epoll 和系统调用，但必须理解 runtime、net/http、context、连接池和 Linux 排障背后的 OS/网络基础。

---

# 第一部分：操作系统

# 一、进程、线程、协程、goroutine

## 【文本块 2】Q：进程、线程、协程有什么区别？

进程是操作系统进行资源分配的基本单位。

一个进程有自己独立的虚拟地址空间、文件描述符表、堆、全局变量、代码段等资源。进程之间隔离性强，一个进程崩溃通常不会直接破坏另一个进程的内存。

线程是 CPU 调度的基本单位。

同一个进程里的多个线程共享进程的地址空间和资源，比如堆、全局变量、文件描述符等。但每个线程有自己的栈、寄存器上下文和执行流。

协程一般指用户态调度的轻量级执行单元。

它不一定由操作系统内核直接调度，而是由语言 runtime 或框架调度。协程切换通常比线程切换轻量，因为很多情况下不需要陷入内核态，也不需要完整的 OS 线程上下文切换。

Go 里的 goroutine 可以理解成 Go runtime 管理的轻量级协程。

它不是 OS 线程，但最终必须运行在 OS 线程上。Go runtime 通过 GMP 调度模型，把大量 goroutine 复用到较少的 OS 线程上执行。

面试里可以这样回答：

进程强调资源隔离，线程强调 CPU 调度和共享进程资源，协程强调用户态轻量调度。Go 的 goroutine 是由 Go runtime 调度的轻量级执行单元，创建成本和切换成本都比 OS 线程低，适合高并发 IO 场景。

---

## 【文本块 3】Go 里的 goroutine 和线程是什么关系？

goroutine 最终要运行在 OS 线程上。

Go runtime 里常说的 GMP：

G 是 goroutine，代表一个待执行的任务。
M 是 machine，基本对应 OS thread。
P 是 processor，代表调度上下文，负责维护本地运行队列。

G 不是直接绑定 M 的。大量 G 会被调度到有限的 M 上执行，而 M 需要绑定 P 才能执行 Go 代码。

这样做的好处是：

1. goroutine 创建成本低
2. goroutine 初始栈很小，可以动态增长
3. 网络 IO 阻塞时，runtime 可以挂起 goroutine，而不是长期占用线程
4. 大量并发连接不需要大量 OS 线程
5. 调度器可以做 work stealing，提高 CPU 利用率

面试里可以补一句：

Go 不是“一个请求一个线程”，而更像是“一个请求一个 goroutine，goroutine 由 runtime 复用到 OS 线程上执行”。

---

## 【代码块 1】观察 goroutine 数量

```go
package main

import (
	"fmt"
	"runtime"
	"time"
)

func main() {
	for i := 0; i < 1000; i++ {
		go func() {
			time.Sleep(time.Second)
		}()
	}

	fmt.Println("goroutines:", runtime.NumGoroutine())

	time.Sleep(2 * time.Second)
	fmt.Println("goroutines after sleep:", runtime.NumGoroutine())
}
```

---

## 【文本块 4】代码解释

这个例子启动了 1000 个 goroutine。

如果用 OS 线程直接承载 1000 个并发任务，成本会比较高；但 goroutine 比线程轻很多，所以 Go 可以比较自然地写出高并发代码。

不过要注意：

goroutine 轻量不等于免费。

如果 goroutine 因为 channel、锁、网络 IO、select、context 没处理好而无法退出，就会造成 goroutine 泄漏。

---

## 【文本块 5】Q：goroutine 泄漏是什么？怎么排查？

goroutine 泄漏是指 goroutine 已经没有业务意义，但仍然因为阻塞在某个地方无法退出。

常见原因：

1. channel 读阻塞，没有人写
2. channel 写阻塞，没有人读
3. context 没有传递取消信号
4. 后台循环没有退出条件
5. HTTP/RPC 调用没有超时
6. select 忘了监听 ctx.Done()
7. worker pool 没有关闭任务队列
8. 定时器、ticker 没 Stop

排查方式：

1. 看 `runtime.NumGoroutine()`
2. 开启 pprof
3. 访问 `/debug/pprof/goroutine?debug=2`
4. 看大量 goroutine 是否卡在相同栈
5. 结合日志和 trace_id 定位创建位置

Go 项目里最重要的预防方式是：

所有可能长期阻塞的 goroutine，都要能被 context、channel close 或超时机制打断。

---

## 【代码块 2】goroutine 泄漏示例

```go
package main

import (
	"fmt"
	"runtime"
	"time"
)

func leak() {
	ch := make(chan int)

	go func() {
		// 永远等不到数据，goroutine 泄漏
		<-ch
	}()
}

func main() {
	for i := 0; i < 1000; i++ {
		leak()
	}

	time.Sleep(100 * time.Millisecond)
	fmt.Println("goroutines:", runtime.NumGoroutine())
}
```

---

## 【代码块 3】用 context 避免 goroutine 泄漏

```go
package main

import (
	"context"
	"fmt"
	"runtime"
	"time"
)

func worker(ctx context.Context) {
	go func() {
		for {
			select {
			case <-ctx.Done():
				return
			case <-time.After(100 * time.Millisecond):
				// do something
			}
		}
	}()
}

func main() {
	ctx, cancel := context.WithCancel(context.Background())

	for i := 0; i < 1000; i++ {
		worker(ctx)
	}

	time.Sleep(100 * time.Millisecond)
	fmt.Println("before cancel:", runtime.NumGoroutine())

	cancel()

	time.Sleep(200 * time.Millisecond)
	fmt.Println("after cancel:", runtime.NumGoroutine())
}
```

---

# 二、用户态、内核态、系统调用

## 【文本块 6】Q：什么是用户态和内核态？

用户态是应用程序运行的受限环境。

在用户态下，程序不能直接操作硬件，不能直接访问任意物理内存，也不能直接操作网卡、磁盘、进程调度等底层资源。

内核态是操作系统内核运行的高权限环境。

内核可以管理 CPU、内存、磁盘、网络、文件系统、进程调度等底层资源。

为什么要区分用户态和内核态？

因为如果应用程序可以随意操作硬件和内存，一个 bug 就可能把整个系统搞崩。通过权限隔离，操作系统可以保护系统稳定性和进程之间的安全边界。

Go 程序平时运行在用户态。
当它要读写文件、建立网络连接、申请某些系统资源时，就需要通过系统调用进入内核态，由内核完成操作。

面试回答：

用户态权限低，应用代码运行在用户态；内核态权限高，操作系统内核运行在内核态。应用程序不能直接操作硬件和系统资源，需要通过系统调用切换到内核态，让内核代为完成文件 IO、网络 IO、进程管理等操作。

---

## 【文本块 7】Q：系统调用、中断、异常有什么区别？

这三个概念都可能导致 CPU 从用户态切到内核态，但触发来源不同。

系统调用是应用程序主动请求内核服务。

比如：

1. read
2. write
3. open
4. close
5. socket
6. connect
7. accept
8. mmap
9. clone
10. futex

Go 里的网络、文件、锁阻塞、内存映射等底层最终都可能涉及系统调用。

中断通常来自外部设备或时钟。

比如网卡收到数据、磁盘 IO 完成、定时器中断，CPU 会暂停当前执行流，转去执行中断处理程序。

异常通常是 CPU 执行当前指令时发生的特殊事件。

比如缺页异常、除零错误、非法内存访问等。

面试里可以这样区分：

系统调用是应用主动找内核帮忙。
中断是外部设备或时钟打断 CPU。
异常是当前程序执行过程中出现特殊情况。

最简单的判断角度是：谁触发、为什么切换。

---

## 【文本块 8】Q：上下文切换是什么？为什么它贵？

上下文切换是 CPU 从一个执行单元切换到另一个执行单元前，需要保存当前执行状态，再恢复另一个执行状态。

要保存和恢复的内容包括：

1. 程序计数器
2. 寄存器
3. 栈指针
4. 线程状态
5. 内存映射相关信息
6. 调度器元数据

上下文切换贵，主要有三个原因：

第一，保存和恢复寄存器、栈、程序计数器本身有成本。

第二，线程切换可能破坏 CPU 缓存局部性。

原来线程的数据还在 L1/L2 cache 里，切到另一个线程后，缓存命中率下降。

第三，进程切换比线程切换更重。

因为进程有独立地址空间，切换进程可能导致页表切换、TLB 失效，后续内存访问成本变高。

Go 里的 goroutine 切换比 OS 线程切换轻，但也不是没有成本。

如果程序创建了大量 goroutine，并且频繁阻塞、唤醒、抢锁，也会有调度成本。

面试回答：

上下文切换的本质是保存当前执行现场并恢复另一个执行现场。它贵不只在保存寄存器，还在于缓存局部性变差、进程切换时地址空间和 TLB 可能失效。高性能系统不是线程越多越好，频繁切换本身就是成本。

---

# 三、虚拟内存、分页、页表、TLB

## 【文本块 9】Q：操作系统为什么需要虚拟内存？

虚拟内存可以理解为：程序看到的是连续的虚拟地址空间，操作系统和 MMU 负责把虚拟地址映射到真实物理内存。

它的价值主要有三个：

第一，进程隔离。

每个进程都有自己的虚拟地址空间。进程 A 不能随便访问进程 B 的内存，提高安全性和稳定性。

第二，简化程序对内存的理解。

程序看到的是从某个地址开始的连续空间，不需要关心真实物理内存是不是碎片化。

第三，提高内存利用率。

操作系统可以把不常用的页换出到磁盘，需要时再换入。也可以通过按需分页，让程序启动时不用一次性加载所有内容。

面试回答：

虚拟内存的核心是把程序使用的虚拟地址映射到物理地址。它解决了进程隔离、地址空间抽象和内存扩展的问题，让程序不用直接面对复杂的物理内存布局。

---

## 【文本块 10】Q：分页和分段有什么区别？

分页是把虚拟地址空间和物理内存切成固定大小的页。

比如常见页大小是 4KB。虚拟地址通过页表映射到物理页框。

分页的优势是：

1. 固定大小，便于管理
2. 减少外部碎片
3. 适合虚拟内存和物理内存映射
4. 方便按需加载和页面置换

分段是按程序逻辑模块划分。

比如代码段、数据段、堆段、栈段。

分段更符合程序逻辑，但段大小不固定，容易产生外部碎片。

现代操作系统里，分页更核心。很多系统会结合分段和分页，但面试里重点讲分页即可。

---

## 【文本块 11】Q：虚拟地址怎么转换成物理地址？

大致流程是：

1. CPU 发出虚拟地址
2. MMU 根据虚拟页号查页表
3. 找到对应的物理页框号
4. 再加上页内偏移
5. 得到最终物理地址

虚拟地址可以拆成两部分：

高位是页号。
低位是页内偏移。

页表负责记录：

虚拟页 -> 物理页框

如果每次访存都去内存里查页表，成本很高，所以 CPU 里有 TLB。

---

## 【文本块 12】Q：TLB 是什么？

TLB 可以理解成页表的高速缓存。

它缓存最近使用过的虚拟页到物理页框的映射。

如果 TLB 命中，地址转换很快。
如果 TLB 未命中，就要去内存里查页表，成本更高。

进程切换时，由于不同进程地址空间不同，TLB 里的映射可能失效，所以进程切换比线程切换更重。

面试回答：

页表负责虚拟地址到物理地址的映射，TLB 是页表缓存。TLB 命中能减少地址转换成本；进程切换可能导致 TLB 失效，所以频繁进程切换会影响性能。

---

## 【文本块 13】Q：什么是缺页中断？

缺页中断是指程序访问的虚拟页合法，但当前不在物理内存中。

这时 CPU 触发缺页异常，进入内核态，由操作系统处理：

1. 判断地址是否合法
2. 如果非法，触发段错误
3. 如果合法但页不在内存，从磁盘加载到内存
4. 更新页表
5. 恢复程序继续执行

缺页不一定是错误。

比如程序第一次访问某个还没加载的页，操作系统会按需加载。

但是如果频繁缺页，说明内存压力大，可能发生 swap，性能会非常差。

Go 服务如果内存占用过高，触发系统 swap，接口 RT 会明显抖动。因此生产环境通常要避免服务进程把机器内存打满。

---

# 四、堆、栈、mmap、Page Cache

## 【文本块 14】Q：堆和栈有什么区别？

栈主要用于函数调用。

它保存函数参数、返回地址、局部变量、调用上下文等。栈通常自动分配和释放，函数调用入栈，函数返回出栈。

堆用于动态分配对象。

对象生命周期不一定和函数调用一致，需要由程序或垃圾回收器管理。

在 Go 里，变量不一定因为写在函数里就分配在栈上，也不一定因为用了 new 就一定在堆上。Go 编译器会做逃逸分析。

如果一个变量的生命周期超过当前函数，或者被外部引用，就可能逃逸到堆上。

面试回答：

栈适合管理函数调用上下文，分配释放快；堆适合管理生命周期更灵活的对象，需要 GC 管理。Go 里变量分配在栈还是堆，不看语法表面，而看逃逸分析结果。

---

## 【代码块 4】查看 Go 逃逸分析

```go
package main

type User struct {
	Name string
}

func NewUser(name string) *User {
	u := User{Name: name}
	return &u
}

func main() {
	_ = NewUser("Tom")
}
```

---

## 【文本块 15】运行方式说明

在终端里执行：

```bash
go build -gcflags="-m" main.go
```

可以看到编译器的逃逸分析输出。

如果局部变量返回了指针，通常会逃逸到堆上。Go 会保证这个指针有效，所以“返回局部变量指针”在 Go 里不是悬垂指针问题。

---

## 【文本块 16】Q：Page Cache 是什么？

Page Cache 是操作系统用内存缓存磁盘文件内容的机制。

当程序读文件时，数据可能先进入 Page Cache。后续再次读取同一文件，如果命中 Page Cache，就不需要真的访问磁盘。

当程序写文件时，也可能先写到 Page Cache，再由操作系统异步刷盘。

Page Cache 对数据库、日志、消息队列都很重要。

比如 Kafka 高吞吐很大程度依赖顺序写和 Page Cache。
MySQL 也有自己的 Buffer Pool，但操作系统层面仍然存在 Page Cache。
Go 服务写日志时，也会受到 Page Cache 和磁盘刷写策略影响。

面试回答：

Page Cache 是 OS 用内存缓存磁盘页的机制，能显著减少磁盘 IO。读文件时可能命中缓存，写文件时也可能先写缓存再异步刷盘。高性能存储系统经常利用 Page Cache 提升吞吐。

---

## 【文本块 17】Q：mmap 是什么？

mmap 是一种内存映射机制，可以把文件映射到进程虚拟地址空间。

程序像访问内存一样访问文件内容，操作系统负责把对应页面加载到内存，并在需要时写回文件。

mmap 的好处：

1. 减少 read/write 的用户态和内核态拷贝
2. 适合随机访问大文件
3. 可以让文件内容按需加载
4. 多进程可以通过映射同一文件共享数据

但 mmap 也有风险：

1. 缺页时可能触发 page fault
2. 写回时机需要理解
3. 文件截断可能导致异常
4. 不适合所有 IO 场景

Go 标准库没有直接提供高级 mmap API，通常会使用第三方库或 syscall。

---

# 五、死锁

## 【文本块 18】Q：什么是死锁？

死锁是指多个线程或 goroutine 互相等待对方释放资源，导致所有参与方都无法继续执行。

本质是形成了循环等待。

例如：

goroutine A 持有锁 1，等待锁 2。
goroutine B 持有锁 2，等待锁 1。

双方都等对方释放，最终卡死。

---

## 【文本块 19】Q：死锁的四个必要条件是什么？

死锁有四个必要条件：

第一，互斥。
资源同一时间只能被一个执行单元持有。

第二，请求并保持。
已经持有资源的同时，又请求新的资源。

第三，不可剥夺。
资源不能被强行抢走，只能由持有者主动释放。

第四，循环等待。
多个执行单元形成环形等待关系。

只要破坏其中任意一个条件，就可以避免死锁。

面试里最常用的工程手段是：

固定加锁顺序。

比如所有地方都先锁用户 ID 小的，再锁用户 ID 大的，避免循环等待。

---

## 【代码块 5】Go 死锁示例

```go
package main

import (
	"fmt"
	"sync"
)

func main() {
	var mu sync.Mutex

	mu.Lock()
	fmt.Println("locked once")

	// 同一个 goroutine 再次 Lock，sync.Mutex 不可重入，死锁
	mu.Lock()
	fmt.Println("locked twice")
}
```

---

## 【文本块 20】代码解释

Go 的 `sync.Mutex` 不是可重入锁。

同一个 goroutine 已经持有锁后，再次 Lock 会阻塞自己，导致死锁。

所以 Go 里要尽量保持加锁范围小、逻辑简单，不要在锁内调用复杂函数，更不要在锁内调用可能再次加锁的函数。

---

## 【代码块 6】固定加锁顺序避免死锁

```go
package main

import (
	"fmt"
	"sync"
)

type Account struct {
	ID      int64
	Balance int64
	Mu      sync.Mutex
}

func Transfer(a, b *Account, amount int64) {
	first, second := a, b
	if first.ID > second.ID {
		first, second = second, first
	}

	first.Mu.Lock()
	defer first.Mu.Unlock()

	second.Mu.Lock()
	defer second.Mu.Unlock()

	a.Balance -= amount
	b.Balance += amount
}

func main() {
	a := &Account{ID: 1, Balance: 100}
	b := &Account{ID: 2, Balance: 100}

	Transfer(a, b, 10)

	fmt.Println(a.Balance, b.Balance)
}
```

---

## 【文本块 21】Go 中避免死锁的工程建议

第一，固定加锁顺序。

多个锁同时使用时，必须全局约定顺序。

第二，缩短持锁时间。

锁内只做内存状态修改，不做网络 IO、数据库查询、日志大批量写入。

第三，避免锁内调用外部函数。

外部函数可能再次加锁，或者执行不可控耗时逻辑。

第四，使用 defer 解锁。

简单场景下可读性更好，但极端热点路径可以评估手动 unlock。

第五，必要时用 try lock 或 context 控制。

Go 标准库 Mutex 没有传统 TryLock 很长时间；新版本支持 TryLock，但不建议滥用。更多时候应通过设计避免复杂锁竞争。

第六，channel 也可能死锁。

不要以为不用锁就不会死锁。无缓冲 channel 双方不匹配，也会卡死。

---

# 六、IO 模型和 IO 多路复用

## 【文本块 22】Q：常见 IO 模型有哪些？

常见可以分成：

第一，阻塞 IO。

调用 read 时，如果数据没准备好，线程就阻塞等待。

第二，非阻塞 IO。

调用 read 时，如果数据没准备好，立即返回 EAGAIN，不阻塞线程。应用需要反复轮询。

第三，IO 多路复用。

一个线程通过 select、poll、epoll 同时监听多个 fd，哪个 fd 就绪，就处理哪个。

第四，信号驱动 IO。

内核在 fd 就绪时发信号通知应用，较少作为主流后端模型讨论。

第五，异步 IO。

应用提交 IO 请求后立即返回，内核完成后通知应用。

面试里常用组合：

BIO：同步阻塞。
NIO/IO 多路复用：同步非阻塞。
AIO：异步非阻塞。

要注意：

IO 多路复用不是异步 IO。
它更准确地说是同步非阻塞的事件驱动模型。

因为应用拿到就绪事件后，仍然要自己调用 read/write 完成数据读写。

---

## 【文本块 23】Q：什么是 IO 多路复用？

IO 多路复用的核心思想是：

一个线程可以同时监听多个文件描述符的 IO 事件。

传统阻塞模型下，如果每个连接一个线程，连接数一多，线程数会爆炸，上下文切换和内存开销都很大。

IO 多路复用让一个线程或少量线程管理大量连接。只有当某个连接真的可读、可写时，才处理它。

典型实现：

1. select
2. poll
3. epoll

面试回答：

IO 多路复用就是一个线程同时监听多个 fd，当某些 fd 就绪时再处理它们。它解决的是高并发连接场景下线程数量过多的问题，典型实现有 select、poll、epoll，其中 Linux 高并发场景最常用 epoll。

---

## 【文本块 24】Q：select、poll、epoll 有什么区别？

select：

1. 使用位图描述 fd 集合
2. 有 fd 数量限制
3. 每次调用都要把 fd 集合从用户态拷贝到内核态
4. 返回后还要遍历所有 fd 判断谁就绪

poll：

1. 使用数组或链表描述 fd
2. 没有 select 那种固定数量上限
3. 但仍然需要把 fd 集合传入内核
4. 返回后仍然要遍历所有 fd

epoll：

1. 使用事件驱动
2. 通过 epoll_ctl 注册 fd
3. 通过 epoll_wait 等待就绪事件
4. 不需要每次全量传递 fd 集合
5. 返回的是就绪 fd，避免全量扫描
6. 大量连接、少量活跃场景下性能更好

面试回答：

select 有数量限制且需要全量拷贝和全量遍历；poll 解决了数量限制，但仍然要遍历；epoll 使用事件驱动，提前注册 fd，epoll_wait 返回就绪事件，因此更适合高并发连接场景。

---

## 【文本块 25】Q：LT 和 ET 有什么区别？

epoll 有两种触发模式：

LT，Level Triggered，水平触发。
只要 fd 还有数据没读完，epoll_wait 就会继续通知。

ET，Edge Triggered，边缘触发。
只有状态从不可读变成可读的那一刻通知一次。如果这次没读完，后续可能不会再通知。

LT 更安全，不容易写错。
ET 性能更高，但要求一次性把数据读到 EAGAIN，否则可能丢事件。

面试回答：

LT 是只要条件满足就反复通知，编程简单；ET 是状态变化时通知一次，性能更好但要求把数据一次读干净，否则容易漏读。工程里 ET 通常要配合非阻塞 IO 循环读到 EAGAIN。

---

# 七、Go netpoller

## 【文本块 26】Q：Go 网络 IO 是阻塞还是非阻塞？

从代码写法看，Go 网络 IO 是阻塞风格。

例如：

```go
n, err := conn.Read(buf)
```

如果没有数据，这个 goroutine 看起来会阻塞。

但从 runtime 底层看，Go 使用非阻塞 fd + netpoller。

当 goroutine 读网络连接发现数据没准备好时，runtime 会把当前 goroutine 挂起，交给 netpoller 等待网络事件。OS 线程不会一直傻等这个连接，而是可以继续执行其他 goroutine。

等 fd 就绪后，netpoller 再把对应 goroutine 放回可运行队列。

所以 Go 的优势是：

应用层代码像阻塞 IO 一样简单。
runtime 底层像事件驱动一样高效。

面试回答：

Go 网络编程表面是阻塞式 Read/Write，但 runtime 底层把 socket 设置成非阻塞，并通过 netpoller 监听事件。goroutine 等网络 IO 时会被挂起，不会长期占用 OS 线程，因此 Go 可以用 goroutine-per-connection 的写法支撑高并发连接。

---

## 【代码块 7】Go TCP Echo Server

```go
package main

import (
	"bufio"
	"fmt"
	"net"
)

func handleConn(conn net.Conn) {
	defer conn.Close()

	reader := bufio.NewReader(conn)

	for {
		line, err := reader.ReadString('\n')
		if err != nil {
			return
		}

		_, _ = conn.Write([]byte("echo: " + line))
	}
}

func main() {
	ln, err := net.Listen("tcp", ":9000")
	if err != nil {
		panic(err)
	}
	defer ln.Close()

	fmt.Println("listening on :9000")

	for {
		conn, err := ln.Accept()
		if err != nil {
			continue
		}

		go handleConn(conn)
	}
}
```

---

## 【文本块 27】代码解释

这段代码是典型的 Go 网络模型：

一个连接一个 goroutine。

如果是传统 OS 线程模型，大量连接会导致线程爆炸。
但 Go 里 goroutine 更轻量，并且网络等待会交给 runtime netpoller，所以这种写法非常常见。

不过生产里还要补充：

1. 连接超时
2. 最大连接数限制
3. 心跳
4. 连接关闭
5. 粘包拆包
6. panic recover
7. 优雅停机
8. 监控连接数和 goroutine 数

---

## 【代码块 8】给 TCP 连接设置读超时

```go
package main

import (
	"bufio"
	"net"
	"time"
)

func handleConn(conn net.Conn) {
	defer conn.Close()

	reader := bufio.NewReader(conn)

	for {
		_ = conn.SetReadDeadline(time.Now().Add(30 * time.Second))

		line, err := reader.ReadString('\n')
		if err != nil {
			return
		}

		_, _ = conn.Write([]byte("echo: " + line))
	}
}
```

---

## 【文本块 28】为什么要设置 deadline？

如果不设置 deadline，一个连接可能长期不发数据，但也不关闭。

对应 goroutine 会一直阻塞在 Read 上，连接 fd 也一直占着。

在长连接系统里，必须有：

1. TCP KeepAlive
2. 应用层心跳
3. 读写 deadline
4. 空闲连接清理

否则很容易出现连接泄漏和 goroutine 泄漏。

---

# 八、Reactor 和 Go 网络模型

## 【文本块 29】Q：IO 多路复用和 Reactor 有什么关系？

IO 多路复用提供事件监听能力。

比如 epoll 告诉你：

哪个 fd 可读。
哪个 fd 可写。
哪个 fd 出错。

Reactor 是事件分发模型。

它负责把这些就绪事件分发给对应的 handler 处理。

可以这么理解：

epoll 负责发现事件。
Reactor 负责分发事件。

Java NIO、Netty、Nginx、Redis 都经常讲 Reactor，因为它们需要显式组织事件循环和 handler。

Go 的普通业务开发很少直接写 Reactor，因为 Go runtime 和 net/http 已经把这些复杂性封装掉了。

但底层思想仍然相关：

Go runtime netpoller 类似负责监听 fd 事件。
Go scheduler 负责把等待 IO 的 goroutine 重新调度起来。
业务代码只需要写阻塞风格的 Read/Write。

面试回答：

IO 多路复用是底层事件监听机制，Reactor 是上层事件分发模型。Go 普通网络代码不需要手写 Reactor，因为 runtime netpoller 和 goroutine 调度器已经把事件监听和调度封装起来了。

---

# 九、零拷贝

## 【文本块 30】Q：什么是零拷贝？

零拷贝不是完全没有拷贝，而是减少 CPU 参与的数据拷贝，尤其是减少用户态和内核态之间的数据复制。

传统文件发送流程：

1. 磁盘 -> 内核缓冲区，DMA 拷贝
2. 内核缓冲区 -> 用户缓冲区，CPU 拷贝
3. 用户缓冲区 -> socket 缓冲区，CPU 拷贝
4. socket 缓冲区 -> 网卡，DMA 拷贝

这里有两次 CPU 拷贝。

使用 sendfile 这类零拷贝机制后，数据可以从内核文件缓冲区直接发送到 socket/网卡路径，避免经过用户态缓冲区，减少 CPU 拷贝和上下文切换。

零拷贝解决的是数据搬运效率问题。
IO 多路复用解决的是连接事件管理问题。

不要混为一谈。

面试回答：

零拷贝关注数据从磁盘到网络的搬运路径，减少用户态和内核态之间的拷贝，降低 CPU 开销；IO 多路复用关注多个连接的事件监听和调度，它们解决的问题不同，但都常出现在高性能网络系统中。

---

## 【文本块 31】Go 里哪里会用到零拷贝？

Go 标准库里，某些场景下 `io.Copy` 可能利用底层优化，例如文件到网络连接时可能走 sendfile 路径。

例如 HTTP 静态文件服务：

```go
http.ServeFile(w, r, "large-file.dat")
```

底层在合适平台和条件下可能使用更高效的数据发送路径。

不过业务面试里不用夸大“Go 一定零拷贝”。更稳的说法是：

Go 标准库会在某些文件传输场景下利用操作系统能力优化拷贝路径，但是否触发和平台、类型、TLS、包装层有关。

尤其要注意：

HTTPS/TLS 场景下，因为数据要加密，sendfile 这类零拷贝路径可能受限。

---

# 十、CPU 缓存、缓存行、伪共享

## 【文本块 32】Q：为什么 CPU 前面有多级缓存？

因为 CPU 计算速度远快于主存访问速度。

如果每次读写数据都直接访问内存，CPU 会大量等待。

所以现代 CPU 一般有多级缓存：

L1 cache：最快，容量小。
L2 cache：稍慢，容量更大。
L3 cache：更大，通常多个核心共享。
主存：容量大，但慢很多。

缓存利用了局部性原理：

时间局部性：刚访问过的数据，很可能很快再次访问。
空间局部性：访问某个地址后，附近地址也很可能被访问。

所以连续遍历数组通常比链表快。
因为数组内存连续，更容易命中缓存行；链表节点分散，容易 cache miss。

---

## 【文本块 33】Q：什么是缓存行？

缓存行是 CPU cache 和内存之间数据传输的基本单位。

常见缓存行大小是 64 字节。

CPU 不会只加载一个 int，而是加载它所在的一整条缓存行。

这对性能有两个影响：

第一，连续内存访问很快。

比如遍历 slice，比遍历链表更容易命中缓存。

第二，可能出现伪共享。

不同线程修改不同变量，但这些变量位于同一缓存行里，会导致缓存行在 CPU 核之间频繁失效和同步。

---

## 【文本块 34】Q：什么是伪共享？

伪共享是指多个线程或 goroutine 修改的是不同变量，但这些变量刚好在同一个缓存行里。

CPU 缓存一致性协议以缓存行为单位工作。

即使变量不同，只要在同一个缓存行中，一个核心修改其中一个变量，就可能导致其他核心对应缓存行失效。

结果是：

看起来没有共享同一个变量，却因为共享同一个缓存行导致性能下降。

面试回答：

伪共享的重点不是逻辑变量是否相同，而是物理上是否落在同一缓存行。多个核心频繁写同一缓存行中的不同变量，会导致缓存行不断失效和同步，从而拖慢程序。

---

## 【代码块 9】用 padding 缓解伪共享

```go
package main

import (
	"fmt"
	"sync/atomic"
	"time"
)

type Counter struct {
	Value int64
}

type PaddedCounter struct {
	Value int64
	_     [56]byte // 假设缓存行 64 字节，int64 占 8 字节
}

func main() {
	var c1 Counter
	var c2 PaddedCounter

	atomic.AddInt64(&c1.Value, 1)
	atomic.AddInt64(&c2.Value, 1)

	fmt.Println(c1.Value, c2.Value)
	time.Sleep(time.Millisecond)
}
```

---

## 【文本块 35】代码解释

padding 的目的不是改变业务语义，而是让不同热点变量尽量落在不同缓存行里，减少伪共享。

在 Go 标准库和 runtime 里，也能看到类似 cache line padding 的设计。

不过业务代码里不要过早优化。只有在高并发计数器、队列、调度器、内存池这类极端热点结构中，伪共享才是重点排查对象。

---

## 【文本块 36】MESI 是什么？

MESI 是一种经典缓存一致性协议。

它定义缓存行的四种状态：

M，Modified，已修改。
当前核心修改过，和主存不一致，其他核心没有有效副本。

E，Exclusive，独占。
当前核心独占该缓存行，和主存一致。

S，Shared，共享。
多个核心可能都有这个缓存行副本，和主存一致。

I，Invalid，失效。
当前缓存行无效，不能使用。

MESI 的目标是让多个 CPU 核心缓存同一份内存时，仍然能保持一致性。

面试不用深入状态转换细节，但要知道：

多核 CPU 每个核心有自己的缓存。
如果一个核心改了共享变量，其他核心不能一直读旧缓存。
缓存一致性协议就是为了解决这个问题。

---

# 十一、Linux 常用排障命令

## 【文本块 37】Q：线上问题你常用哪些 Linux 命令？

我会按资源维度回答，而不是散背命令。

CPU：

```bash
top
htop
top -Hp <pid>
ps -ef
ps -mp <pid> -o THREAD,tid,time
```

内存：

```bash
free -m
vmstat
pmap -x <pid>
cat /proc/<pid>/status
```

磁盘：

```bash
df -h
du -sh *
iostat -x 1
```

网络：

```bash
ss -lntp
ss -antp
netstat -anp
lsof -i :8080
ping
curl
dig
traceroute
tcpdump
```

日志：

```bash
tail -f app.log
grep "ERROR" app.log
less app.log
journalctl -u service-name
```

Go 专属：

```bash
go tool pprof
go tool trace
curl http://127.0.0.1:6060/debug/pprof/goroutine?debug=2
curl http://127.0.0.1:6060/debug/pprof/heap
curl http://127.0.0.1:6060/debug/pprof/profile?seconds=30
```

面试回答：

我通常先看资源概况，再定位进程、线程、端口，最后看日志和代码栈。CPU 用 top/top -Hp，内存用 free/vmstat/pmap，磁盘用 df/du/iostat，网络用 ss/lsof/curl/tcpdump，Go 服务会结合 pprof 看 CPU、heap、goroutine 和 mutex/block profile。

---

## 【文本块 38】Q：CPU 飙高怎么查？

排查顺序：

第一，用 top 看哪个进程 CPU 高。

```bash
top
```

第二，用 top -Hp 看进程里哪个线程 CPU 高。

```bash
top -Hp <pid>
```

第三，对 Go 服务，直接看 pprof CPU profile。

```bash
curl -o cpu.pprof "http://127.0.0.1:6060/debug/pprof/profile?seconds=30"
go tool pprof cpu.pprof
```

进入 pprof 后：

```text
top
list 函数名
web
```

第四，看是否是：

1. 死循环
2. JSON 编解码过重
3. 正则过重
4. 加密压缩过重
5. GC 压力
6. 锁竞争
7. 日志过量
8. 热点 SQL 或 RPC 重试导致 CPU 放大

Go 服务不要像 Java 那样转 16 进制线程 ID 配 jstack，Go 更推荐 pprof 和 goroutine dump。

---

## 【代码块 10】Go 开启 pprof

```go
package main

import (
	"fmt"
	"net/http"
	_ "net/http/pprof"
)

func main() {
	go func() {
		fmt.Println(http.ListenAndServe("127.0.0.1:6060", nil))
	}()

	select {}
}
```

---

## 【文本块 39】pprof 常用地址

启动后可以访问：

```text
http://127.0.0.1:6060/debug/pprof/
http://127.0.0.1:6060/debug/pprof/goroutine?debug=2
http://127.0.0.1:6060/debug/pprof/heap
http://127.0.0.1:6060/debug/pprof/profile?seconds=30
http://127.0.0.1:6060/debug/pprof/mutex
http://127.0.0.1:6060/debug/pprof/block
```

生产环境不要直接把 pprof 暴露到公网。
建议只监听 localhost，或者加鉴权、内网访问、临时开启。

---

## 【文本块 40】Q：内存异常怎么查？

先看系统层：

```bash
free -m
top
vmstat 1
pmap -x <pid>
cat /proc/<pid>/status
```

再看 Go 层：

```bash
curl -o heap.pprof http://127.0.0.1:6060/debug/pprof/heap
go tool pprof heap.pprof
```

Go 内存问题常见原因：

1. slice 截取大数组导致内存滞留
2. map 只增不删
3. goroutine 泄漏
4. 缓存没有上限
5. time.Ticker 没 Stop
6. 大量 []byte/string 转换
7. JSON 反序列化大对象
8. 连接或响应 body 没 Close
9. channel 堆积任务
10. pprof 看到 GC 压力高

面试回答：

Go 内存问题我会先看系统 RSS 和容器内存，再看 heap profile、goroutine 数和 GC 指标。Go 里常见问题不是传统 malloc/free 泄漏，而是对象仍被引用导致 GC 无法回收，比如 map 缓存、goroutine、slice 底层数组、未关闭响应体等。

---

## 【文本块 41】Q：磁盘满了怎么查？

常用命令：

```bash
df -h
du -sh /*
du -sh /var/log/*
find / -type f -size +1G
lsof | grep deleted
```

磁盘满常见原因：

1. 日志没有滚动
2. 大文件临时文件没清理
3. Docker 镜像和容器日志膨胀
4. MySQL binlog 太多
5. core dump 文件
6. 已删除文件仍被进程占用

最后一种很常见：

文件已经 rm 了，但进程还打开着 fd，空间不会释放。
需要用 lsof 找 deleted 文件，然后重启对应进程或关闭 fd。

---

## 【文本块 42】Q：网络连接异常怎么查？

常用命令：

```bash
ss -lntp
ss -antp
lsof -i :8080
curl -v http://host:port/path
tcpdump -i eth0 port 8080
```

关注连接状态：

ESTABLISHED：已建立连接。
LISTEN：正在监听。
TIME_WAIT：主动关闭方等待旧包消失。
CLOSE_WAIT：对端关闭了连接，本端应用还没 close。
SYN_SENT：客户端正在发起连接。
SYN_RECV：服务端收到 SYN，等待握手完成。

Go 服务里如果 CLOSE_WAIT 很多，通常说明应用没有正确关闭连接。

比如 HTTP client 没有关闭 response body。

---

## 【代码块 11】HTTP response body 必须关闭

```go
package main

import (
	"fmt"
	"io"
	"net/http"
	"time"
)

func main() {
	client := &http.Client{
		Timeout: 3 * time.Second,
	}

	resp, err := client.Get("https://example.com")
	if err != nil {
		fmt.Println("request error:", err)
		return
	}
	defer resp.Body.Close()

	body, err := io.ReadAll(resp.Body)
	if err != nil {
		fmt.Println("read error:", err)
		return
	}

	fmt.Println(len(body))
}
```

---

## 【文本块 43】为什么 resp.Body 必须 Close？

如果不 Close，底层 TCP 连接可能无法复用或释放。

后果：

1. 连接泄漏
2. fd 泄漏
3. CLOSE_WAIT 增多
4. HTTP client 连接池耗尽
5. 请求阻塞等待连接
6. 服务 RT 抖动

如果想复用连接，最好读完 body 再 Close。
如果不关心内容，也可以：

```go
io.Copy(io.Discard, resp.Body)
resp.Body.Close()
```

---

# 第二部分：计算机网络

# 十二、网络分层

## 【文本块 44】Q：为什么网络要分层？

网络通信非常复杂，涉及应用数据、传输可靠性、路由寻址、链路转发、物理信号等问题。

如果不分层，所有协议都耦合在一起，系统会非常难维护和演进。

分层的核心价值是职责隔离：

应用层关心传什么数据。
传输层关心端到端进程通信。
网络层关心跨网络找哪台机器。
链路层关心当前链路上怎么传帧。
物理层关心电信号、光信号、无线信号。

面试回答：

网络分层的本质是把复杂通信过程拆成多个职责清晰的层次。每层只依赖下层提供的能力，并向上层提供服务，从而降低复杂度，提高协议复用性和可演进性。

---

## 【文本块 45】OSI 七层和 TCP/IP 四层怎么对应？

OSI 七层：

1. 应用层
2. 表示层
3. 会话层
4. 传输层
5. 网络层
6. 数据链路层
7. 物理层

TCP/IP 四层：

1. 应用层
2. 传输层
3. 网络层
4. 网络接口层

常见协议对应：

应用层：HTTP、HTTPS、DNS、FTP、SMTP、WebSocket、gRPC。
传输层：TCP、UDP。
网络层：IP、ICMP。
链路层：以太网、ARP。

面试里可以说：

OSI 七层更偏理论模型，TCP/IP 四层更贴近互联网实际落地。

---

## 【文本块 46】Q：MAC、IP、端口、Socket 分别是什么？

MAC 地址用于局域网内链路层寻址。
IP 地址用于网络层跨网络寻址，定位哪台机器。
端口用于传输层定位机器上的哪个进程。
Socket 可以理解为网络通信端点，通常由协议、源 IP、源端口、目标 IP、目标端口组成。

例如一个 TCP 连接由四元组唯一标识：

```text
源 IP + 源端口 + 目标 IP + 目标端口
```

面试回答：

IP 解决主机定位，端口解决进程定位，MAC 解决局域网内下一跳帧转发，Socket 是应用程序进行网络通信的抽象端点。

---

# 十三、TCP 和 UDP

## 【文本块 47】Q：TCP 和 UDP 有什么区别？

TCP：

1. 面向连接
2. 可靠传输
3. 保证顺序
4. 面向字节流
5. 有流量控制
6. 有拥塞控制
7. 头部开销更大
8. 延迟相对更高

UDP：

1. 无连接
2. 不保证可靠
3. 不保证顺序
4. 面向数据报
5. 头部小
6. 延迟低
7. 应用层需要自己处理可靠性

典型场景：

TCP：HTTP、HTTPS、MySQL、Redis、RPC、SSH。
UDP：DNS、音视频、实时游戏、QUIC/HTTP3 底层。

面试回答：

如果业务更关注可靠、有序、正确性，通常用 TCP；如果更关注低延迟、能容忍少量丢包，或者应用层自己实现可靠性，可以用 UDP。

---

## 【文本块 48】Q：DNS 为什么常用 UDP？

因为大多数 DNS 查询报文很小，请求响应简单，UDP 没有建连开销，速度更快。

但 DNS 也会用 TCP。

比如：

1. 响应太大，UDP 放不下
2. 区域传输
3. 某些安全场景
4. DoT/DoH 等新协议形态

面试回答：

DNS 常用 UDP 是为了减少连接建立开销，提高查询速度。但当响应过大或需要更可靠传输时，也会使用 TCP。

---

# 十四、TCP 三次握手

## 【文本块 49】Q：TCP 为什么要三次握手？

TCP 三次握手的目的主要有两个：

第一，确认双方收发能力正常。
客户端确认自己能发、能收。
服务端确认自己能发、能收。

第二，同步初始序列号。
TCP 是可靠有序协议，需要双方同步初始序列号，后续才能做确认、重传和排序。

典型流程：

第一次：客户端发送 SYN，表示请求建立连接，并带上客户端初始序列号 x。

第二次：服务端返回 SYN + ACK，表示收到了客户端 SYN，也同意建立连接，并带上服务端初始序列号 y，同时确认 x+1。

第三次：客户端返回 ACK，确认收到了服务端 SYN，确认 y+1。

面试回答：

三次握手不是形式主义，而是为了让双方都确认对方的发送和接收能力，并同步初始序列号。第三次握手尤其关键，它让服务端确认客户端能收到自己的 SYN+ACK。

---

## 【文本块 50】为什么不能两次握手？

如果只有两次握手：

客户端发 SYN。
服务端回 SYN+ACK 后就认为连接建立。

但服务端无法确认客户端是否真的收到了 SYN+ACK。

如果客户端没有收到，服务端却分配连接资源，就可能造成资源浪费。

另外，历史失效的 SYN 报文也可能导致服务端误建立连接。

第三次握手就是为了解决服务端对客户端接收能力的确认问题。

---

## 【文本块 51】为什么不是四次握手？

因为服务端收到客户端 SYN 后，可以把两件事合并：

1. ACK 客户端的 SYN
2. 发送自己的 SYN

所以第二次握手可以是 SYN+ACK，不需要拆成两次。

三次已经足够确认双方收发能力和同步序列号，四次没有必要。

---

# 十五、TCP 四次挥手

## 【文本块 52】Q：TCP 为什么要四次挥手？

TCP 是全双工协议，双方都可以独立发送数据。

关闭连接时，需要分别关闭两个方向：

第一次：主动关闭方发送 FIN，表示我不再发送数据了。
第二次：被动关闭方返回 ACK，表示我知道你不发了。
第三次：被动关闭方处理完剩余数据后，也发送 FIN。
第四次：主动关闭方返回 ACK，确认对方也关闭。

为什么建立连接可以三次，关闭要四次？

因为建立连接时，服务端的 SYN 和 ACK 可以合并。
关闭连接时，被动关闭方收到 FIN 后，可能还有数据没发完，所以 ACK 和 FIN 通常不能立刻合并。

面试回答：

TCP 是全双工通信，两个方向要分别关闭。被动关闭方收到 FIN 后可能还有数据要发送，所以先 ACK，等数据发完后再 FIN，因此通常需要四次挥手。

---

# 十六、TIME_WAIT 和 CLOSE_WAIT

## 【文本块 53】Q：TIME_WAIT 是什么？为什么需要？

TIME_WAIT 出现在主动关闭连接的一方。

主动关闭方在发送最后一个 ACK 后，会进入 TIME_WAIT，等待 2MSL。

它的作用主要有两个：

第一，确保最后一个 ACK 能让对端收到。

如果最后 ACK 丢了，对端会重发 FIN，主动关闭方还在 TIME_WAIT 就能再次回复 ACK。

第二，避免旧连接的残留报文影响新连接。

等待一段时间，让网络中旧连接的延迟报文自然消失。

面试回答：

TIME_WAIT 是主动关闭方为了可靠关闭和避免旧报文污染新连接而保留的状态。它不是 bug，但如果数量异常多，要看是否短连接过多、连接复用不足或主动关闭压力过大。

---

## 【文本块 54】Q：CLOSE_WAIT 是什么？为什么危险？

CLOSE_WAIT 出现在被动关闭连接的一方。

对端已经发送 FIN，本端也 ACK 了，但本端应用程序还没有调用 close 关闭 socket。

如果 CLOSE_WAIT 很多，通常说明应用层没有正确关闭连接。

Go 里常见原因：

1. HTTP response body 没有 Close
2. TCP 连接读到 EOF 后没有 Close
3. handler 卡住，连接迟迟不释放
4. 连接池使用不当
5. goroutine 泄漏导致连接对象还被引用

面试回答：

TIME_WAIT 多通常是主动关闭多，不一定是 bug；CLOSE_WAIT 多更危险，往往说明应用没有正确 close 连接。Go 服务里最常见原因是 HTTP client 没有关闭 resp.Body。

---

# 十七、TCP 可靠性、滑动窗口、流量控制、拥塞控制

## 【文本块 55】Q：TCP 如何保证可靠传输？

TCP 可靠性主要依赖：

1. 序列号
2. 确认应答 ACK
3. 超时重传
4. 快速重传
5. 校验和
6. 滑动窗口
7. 流量控制
8. 拥塞控制

序列号用于标识字节流位置。
ACK 用于确认已收到的数据。
超时重传用于处理丢包。
滑动窗口允许批量发送，提高吞吐。
流量控制防止打爆接收方。
拥塞控制防止打爆网络。

面试回答：

TCP 可靠性不是靠单一机制，而是序列号、ACK、重传、滑动窗口、流量控制和拥塞控制共同实现的。

---

## 【文本块 56】Q：滑动窗口是什么？

如果 TCP 每发一个包都等 ACK，吞吐会很低。

滑动窗口允许发送方在未收到 ACK 前连续发送多个数据段。

窗口大小表示当前最多能有多少未确认数据在路上。

ACK 返回后，窗口向前滑动，发送方继续发送后面的数据。

滑动窗口提升吞吐，但也要配合流量控制和拥塞控制，不能无限发。

---

## 【文本块 57】Q：流量控制和拥塞控制有什么区别？

流量控制关注接收方。

接收方通过 TCP 窗口告诉发送方：

我还能接收多少数据。

如果接收方处理不过来，窗口变小，发送方就少发，避免把接收方缓冲区打爆。

拥塞控制关注网络。

即使接收方能接收，网络中间链路也可能拥塞。TCP 通过慢启动、拥塞避免、快速重传、快速恢复等机制控制发送速度，避免把网络打爆。

面试回答：

流量控制是端到端的，防止发送方把接收方打爆；拥塞控制是面向网络整体的，防止发送方把网络链路打爆。

---

# 十八、TCP 粘包拆包

## 【文本块 58】Q：什么是 TCP 粘包和拆包？

TCP 是字节流协议，不保留应用层消息边界。

应用层写了两次：

```text
hello
world
```

接收方可能一次读到：

```text
helloworld
```

这叫粘包。

应用层写了一次：

```text
very-long-message
```

接收方可能分多次读到：

```text
very-
long-
message
```

这叫拆包。

粘包拆包不是 Go 的问题，也不是 Netty 的问题，而是 TCP 字节流模型天然带来的问题。

UDP 是数据报协议，会保留报文边界，但不保证可靠性。

---

## 【文本块 59】Q：如何解决 TCP 粘包拆包？

常见三种方案：

第一，固定长度协议。

每条消息固定 N 字节。不足补齐。简单但浪费空间，不灵活。

第二，分隔符协议。

比如每条消息以 `\n` 结尾。适合文本协议，但消息体里要处理转义问题。

第三，长度字段协议。

消息头里写 body 长度，接收方先读固定长度 header，再按长度读取 body。

工程里最常用的是长度字段协议。

例如：

```text
| 4 bytes length | body |
```

Go 里可以用 `io.ReadFull` 保证读够指定字节数。

---

## 【代码块 12】长度字段协议编码

```go
package main

import (
	"bytes"
	"encoding/binary"
	"fmt"
)

func EncodeMessage(body []byte) ([]byte, error) {
	var buf bytes.Buffer

	length := uint32(len(body))

	if err := binary.Write(&buf, binary.BigEndian, length); err != nil {
		return nil, err
	}

	if _, err := buf.Write(body); err != nil {
		return nil, err
	}

	return buf.Bytes(), nil
}

func main() {
	data, err := EncodeMessage([]byte("hello"))
	if err != nil {
		panic(err)
	}

	fmt.Printf("%v\n", data)
}
```

---

## 【代码块 13】长度字段协议解码

```go
package main

import (
	"encoding/binary"
	"fmt"
	"io"
	"net"
)

func ReadMessage(conn net.Conn) ([]byte, error) {
	header := make([]byte, 4)

	if _, err := io.ReadFull(conn, header); err != nil {
		return nil, err
	}

	length := binary.BigEndian.Uint32(header)

	body := make([]byte, length)
	if _, err := io.ReadFull(conn, body); err != nil {
		return nil, err
	}

	return body, nil
}

func handle(conn net.Conn) {
	defer conn.Close()

	for {
		msg, err := ReadMessage(conn)
		if err != nil {
			return
		}

		fmt.Println("message:", string(msg))
	}
}
```

---

## 【文本块 60】工程注意点

长度字段协议还要考虑：

1. 最大包大小限制，防止恶意 length 打爆内存
2. endian 约定
3. 协议版本号
4. message type
5. request id
6. 心跳包
7. 压缩标志
8. 校验和
9. 超时控制
10. 半包和异常连接关闭

生产里不要只写 `make([]byte, length)`，一定要先判断 length 是否超过上限。

---

# 十九、HTTP

## 【文本块 61】Q：HTTP 是什么？

HTTP 是应用层协议，通常运行在 TCP 之上。

它是请求-响应模型：

客户端发请求。
服务端返回响应。

HTTP 请求包括：

1. 请求行
2. 请求头
3. 请求体

HTTP 响应包括：

1. 状态行
2. 响应头
3. 响应体

常见状态码：

200：成功。
301/302：重定向。
400：请求参数错误。
401：未认证。
403：无权限。
404：资源不存在。
429：限流。
500：服务端错误。
502：网关收到错误响应。
503：服务不可用。
504：网关等待上游超时。

面试回答：

HTTP 是应用层请求响应协议，基于方法、URL、Header、Body 表达请求，通过状态码、Header、Body 表达响应。后端排查时要能区分 4xx 是客户端请求问题，5xx 是服务端或网关链路问题。

---

## 【文本块 62】Q：GET 和 POST 有什么区别？

GET 通常用于获取资源，参数常放在 URL query 中。
POST 通常用于提交数据，参数常放在 body 中。

语义上：

GET 应该是安全的、幂等的。
POST 不一定安全，也不一定幂等。

但要注意：

GET 也可以有 body，只是不常用，很多中间件支持不好。
POST 也可以设计成幂等，比如通过幂等 key。
安全和幂等是语义约定，不是 HTTP 物理限制。

面试回答：

GET 和 POST 最大区别不是“参数长度”这种表面点，而是语义。GET 用于获取资源，应该安全幂等；POST 用于提交数据，不天然幂等。实际工程里还要考虑缓存、日志、浏览器行为和接口规范。

---

## 【文本块 63】Q：HTTP/1.1、HTTP/2、HTTP/3 有什么区别？

HTTP/1.1：

1. 默认长连接
2. 支持 pipeline，但实践中队头阻塞问题明显
3. 文本协议
4. 一个 TCP 连接上请求响应复用能力有限

HTTP/2：

1. 二进制分帧
2. 多路复用
3. Header 压缩
4. 一个 TCP 连接上可以并发多个 stream
5. 但仍然基于 TCP，TCP 层丢包会影响所有 stream

HTTP/3：

1. 基于 QUIC
2. QUIC 基于 UDP
3. 连接建立更快
4. 改善 TCP 层队头阻塞
5. 内置 TLS 1.3
6. 支持连接迁移

面试回答：

HTTP/1.1 主要靠长连接提升效率，但复用能力有限；HTTP/2 引入二进制分帧和多路复用，但仍受 TCP 队头阻塞影响；HTTP/3 基于 QUIC/UDP，在传输层改善队头阻塞并支持更快握手和连接迁移。

---

# 二十、Go net/http

## 【文本块 64】Q：Go net/http 服务端模型是什么？

Go 的 `net/http` 服务端大致是：

1. Listen 监听端口
2. Accept 接收 TCP 连接
3. 每个连接由 goroutine 处理
4. 每个请求调用对应 Handler
5. Handler 写 ResponseWriter 返回响应

Go 的 HTTP server 屏蔽了很多底层细节：

1. TCP accept
2. HTTP 解析
3. keep-alive
4. header/body
5. chunked
6. HTTP/2 支持
7. connection close

业务只需要实现：

```go
func(w http.ResponseWriter, r *http.Request)
```

但生产里必须配置超时。

---

## 【代码块 14】Go HTTP Server 推荐配置

```go
package main

import (
	"context"
	"fmt"
	"net/http"
	"time"
)

func main() {
	mux := http.NewServeMux()

	mux.HandleFunc("/ping", func(w http.ResponseWriter, r *http.Request) {
		_, _ = w.Write([]byte("pong"))
	})

	server := &http.Server{
		Addr:              ":8080",
		Handler:           mux,
		ReadHeaderTimeout: 2 * time.Second,
		ReadTimeout:       5 * time.Second,
		WriteTimeout:      10 * time.Second,
		IdleTimeout:       60 * time.Second,
	}

	go func() {
		fmt.Println("listening on :8080")
		if err := server.ListenAndServe(); err != nil && err != http.ErrServerClosed {
			panic(err)
		}
	}()

	time.Sleep(10 * time.Second)

	ctx, cancel := context.WithTimeout(context.Background(), 5*time.Second)
	defer cancel()

	_ = server.Shutdown(ctx)
}
```

---

## 【文本块 65】为什么 HTTP Server 要配置超时？

如果不配置超时，恶意客户端或异常连接可能长期占用连接和 goroutine。

关键参数：

ReadHeaderTimeout：读取请求头超时，防止慢速头攻击。
ReadTimeout：读取完整请求超时。
WriteTimeout：响应写出超时。
IdleTimeout：keep-alive 空闲连接超时。

面试回答：

Go HTTP Server 默认配置不一定适合生产，应该显式设置 ReadHeaderTimeout、ReadTimeout、WriteTimeout、IdleTimeout，避免慢连接或异常客户端长期占用资源。

---

## 【文本块 66】Q：Go HTTP Client 有什么坑？

默认的 `http.Client{}` 没有全局 Timeout。
如果不设置 Timeout，请求可能因为网络异常长期卡住。

另外，默认 Transport 连接池参数不一定适合高并发服务。

生产里要关注：

1. Timeout
2. DialContext timeout
3. TLSHandshakeTimeout
4. ResponseHeaderTimeout
5. IdleConnTimeout
6. MaxIdleConns
7. MaxIdleConnsPerHost
8. MaxConnsPerHost
9. resp.Body.Close
10. 重试和幂等

---

## 【代码块 15】Go HTTP Client 推荐配置

```go
package main

import (
	"net"
	"net/http"
	"time"
)

func NewHTTPClient() *http.Client {
	transport := &http.Transport{
		Proxy: http.ProxyFromEnvironment,

		DialContext: (&net.Dialer{
			Timeout:   2 * time.Second,
			KeepAlive: 30 * time.Second,
		}).DialContext,

		TLSHandshakeTimeout:   2 * time.Second,
		ResponseHeaderTimeout: 3 * time.Second,
		ExpectContinueTimeout: 1 * time.Second,

		MaxIdleConns:        100,
		MaxIdleConnsPerHost: 20,
		MaxConnsPerHost:     50,
		IdleConnTimeout:     90 * time.Second,
	}

	return &http.Client{
		Transport: transport,
		Timeout:   5 * time.Second,
	}
}
```

---

## 【文本块 67】Go HTTP Client 面试回答模板

Go HTTP client 生产里不能直接裸用默认配置。我会设置全局 Timeout，同时配置 Transport 的连接池、Dial 超时、TLS 握手超时、响应头超时和空闲连接超时。每次请求必须关闭 resp.Body，否则连接无法复用，甚至造成 fd 和 goroutine 泄漏。对于写请求重试，还必须考虑幂等。

---

# 二十一、HTTPS 和 TLS

## 【文本块 68】Q：HTTP 和 HTTPS 有什么区别？

HTTPS = HTTP + TLS。

HTTP 明文传输，容易被窃听、篡改和冒充。

HTTPS 通过 TLS 提供：

1. 加密
2. 身份认证
3. 完整性校验

加密防止内容被窃听。
证书防止连接到假服务器。
完整性校验防止数据被篡改。

面试回答：

HTTPS 不是新的应用协议，它本质上是在 HTTP 和 TCP 之间加了一层 TLS，用于提供加密、身份认证和完整性保护。

---

## 【文本块 69】Q：TLS 握手大致过程是什么？

以常见 TLS 1.2 简化理解：

1. 客户端发送 ClientHello，包含支持的 TLS 版本、随机数、加密套件等
2. 服务端返回 ServerHello，选择 TLS 版本和加密套件，返回证书和随机数
3. 客户端验证证书
4. 客户端生成预主密钥，用服务端公钥加密后发送
5. 双方根据随机数和预主密钥生成会话密钥
6. 后续使用对称加密传输 HTTP 数据

TLS 1.3 对握手做了优化，减少往返次数，安全性也更好。

面试回答：

TLS 握手核心是用非对称加密解决身份认证和密钥协商，用协商出的对称密钥加密后续数据。因为对称加密性能更好，所以真正传输数据时不会一直用非对称加密。

---

## 【文本块 70】Q：为什么 HTTPS 不全程用非对称加密？

因为非对称加密计算成本高，性能比对称加密差很多。

HTTPS 的设计是：

握手阶段用非对称加密和证书解决身份认证、密钥协商。
数据传输阶段用对称加密提高性能。

这也是混合加密设计。

---

# 二十二、DNS 和 CDN

## 【文本块 71】Q：DNS 解析过程是什么？

DNS 的作用是把域名解析成 IP 地址。

大致流程：

1. 浏览器缓存
2. 操作系统缓存
3. 本地 hosts
4. 本地 DNS 服务器
5. 根 DNS
6. 顶级域 DNS
7. 权威 DNS
8. 返回 IP

实际不会每次都走完整流程，因为中间有多级缓存。

面试回答：

DNS 解析不是每次都从根 DNS 开始查，浏览器、操作系统、本地 DNS、递归 DNS 都可能缓存结果。排查 DNS 问题时要关注解析结果、TTL、CNAME、权威 DNS 和缓存。

---

## 【文本块 72】Q：CDN 是什么？

CDN 是内容分发网络。

它的核心思想是：

把静态资源或可缓存内容分发到离用户更近的节点，让用户就近访问，降低延迟和源站压力。

典型流程：

1. 用户访问 CDN 域名
2. DNS 调度到合适的 CDN 节点
3. CDN 节点有缓存，直接返回
4. 没有缓存，则回源站拉取
5. 拉取后缓存，再返回用户

CDN 适合：

1. 图片
2. 视频
3. JS/CSS
4. 下载文件
5. 静态 HTML
6. 部分可缓存 API

面试回答：

CDN 通过 DNS 调度和边缘缓存，让用户访问离自己更近的节点。命中缓存时不需要回源，能降低延迟、减少带宽和保护源站。

---

# 二十三、输入 URL 到页面展示

## 【文本块 73】Q：从输入 URL 到页面展示发生了什么？

建议按这条链路回答：

1. URL 解析
2. 浏览器缓存检查
3. DNS 解析
4. 建立 TCP 连接
5. 如果是 HTTPS，进行 TLS 握手
6. 发送 HTTP 请求
7. 服务端处理请求
8. 返回 HTTP 响应
9. 浏览器解析 HTML
10. 构建 DOM
11. 解析 CSS 构建 CSSOM
12. 生成渲染树
13. 布局
14. 绘制
15. 加载 JS、CSS、图片、字体等后续资源
16. 前端框架执行和接口请求

如果结合后端项目，可以补充：

请求到达网关或负载均衡。
网关做 TLS 终止、路由、鉴权、限流。
后端 Go 服务处理请求。
服务查询 Redis/MySQL/RPC。
返回 JSON 或 HTML。
浏览器继续渲染页面。

面试回答：

从输入 URL 到页面展示，本质上是先通过 DNS 找到目标 IP，再通过 TCP/TLS 建立连接，然后发送 HTTP 请求，服务端处理并返回响应，最后浏览器解析 HTML/CSS/JS 并完成布局绘制。

---

# 二十四、网络排查

## 【文本块 74】Q：线上接口访问慢或失败，网络怎么排查？

我会按链路逐层缩小范围：

第一，DNS。

```bash
dig example.com
nslookup example.com
```

看域名解析是否正确，TTL 是否符合预期，是否解析到了错误 IP。

第二，连通性。

```bash
ping ip
traceroute ip
```

ping 看粗略连通性、延迟和丢包。
traceroute 看路径中哪一跳开始异常。

第三，端口。

```bash
ss -lntp
telnet host port
nc -vz host port
```

看端口是否监听、防火墙是否拦截、连接是否建立。

第四，HTTP/HTTPS。

```bash
curl -v https://example.com/api
curl -k -v https://ip/api -H "Host: example.com"
```

看状态码、响应头、TLS、跳转、耗时。

第五，连接状态。

```bash
ss -antp
```

重点看 SYN_SENT、SYN_RECV、ESTABLISHED、TIME_WAIT、CLOSE_WAIT。

第六，抓包。

```bash
tcpdump -i eth0 host 1.2.3.4 and port 443
```

看包是否发出、是否收到响应、是否重传。

第七，应用日志。

看 Go 服务日志、trace_id、错误栈、下游调用耗时、连接池状态。

面试回答：

网络排查的核心不是背命令，而是按 DNS、连通性、端口、协议、应用这一条链逐层缩小范围。先确认域名解析，再确认网络通不通，再看端口和 TCP 状态，再用 curl 看 HTTP/TLS，最后结合应用日志和抓包定位。

---

## 【文本块 75】Q：ping 通代表服务正常吗？

不代表。

ping 使用 ICMP，只能说明目标主机在网络层可能可达。

但应用服务还依赖：

1. TCP 端口是否监听
2. 防火墙是否放行
3. 应用是否正常
4. HTTP 路由是否正确
5. TLS 证书是否正常
6. 网关是否转发
7. 后端依赖是否可用

所以 ping 通不代表接口能访问。
ping 不通也不一定代表服务挂了，可能禁用了 ICMP。

---

## 【文本块 76】Q：curl 在排查中有什么价值？

curl 可以直接验证 HTTP/HTTPS 层。

常用方式：

```bash
curl -v https://example.com/api
curl -I https://example.com
curl -X POST https://example.com/api -H "Content-Type: application/json" -d '{"a":1}'
curl --connect-timeout 2 --max-time 5 https://example.com
```

可以看：

1. DNS 结果
2. TCP 连接
3. TLS 握手
4. HTTP 状态码
5. 响应头
6. 响应体
7. 跳转
8. 总耗时

如果怀疑 DNS 问题，可以直接打 IP 并指定 Host：

```bash
curl -v https://1.2.3.4/api -H "Host: example.com"
```

---

## 【文本块 77】Q：502、503、504 有什么区别？

502 Bad Gateway：网关收到了上游的错误响应，或者上游返回非法响应。

常见原因：

1. 上游服务崩溃
2. 上游返回格式异常
3. 网关到上游连接失败
4. 协议不匹配

503 Service Unavailable：服务不可用。

常见原因：

1. 服务过载
2. 被限流
3. 正在发布或维护
4. 上游实例全部不可用

504 Gateway Timeout：网关等待上游超时。

常见原因：

1. 上游处理太慢
2. 数据库慢
3. RPC 超时
4. 网络抖动
5. 网关超时配置太短

面试回答：

502 更像“上游返回错了或连不上”，503 更像“服务当前不接单”，504 更像“上游太慢，网关等超时”。排查时要结合网关日志、上游服务日志和 trace 链路耗时。

---

# 二十五、Go 项目里的超时、重试、连接复用

## 【文本块 78】Q：Go 项目里为什么一定要设置超时？

因为没有超时，就没有资源释放边界。

一个下游慢请求可能长期占用：

1. goroutine
2. TCP 连接
3. HTTP 连接池
4. MySQL 连接
5. Redis 连接
6. 内存
7. 上游请求上下文

如果流量一大，就会造成级联阻塞。

Go 里应该把 context 从请求入口一路传下去：

HTTP handler -> service -> repository -> MySQL/Redis/RPC/HTTP client

每一层都能感知取消和超时。

---

## 【代码块 16】HTTP handler 中传递 context

```go
package main

import (
	"context"
	"fmt"
	"net/http"
	"time"
)

func queryDB(ctx context.Context) error {
	select {
	case <-time.After(2 * time.Second):
		return nil
	case <-ctx.Done():
		return ctx.Err()
	}
}

func handler(w http.ResponseWriter, r *http.Request) {
	ctx, cancel := context.WithTimeout(r.Context(), time.Second)
	defer cancel()

	if err := queryDB(ctx); err != nil {
		http.Error(w, err.Error(), http.StatusGatewayTimeout)
		return
	}

	_, _ = w.Write([]byte("ok"))
}

func main() {
	http.HandleFunc("/demo", handler)
	fmt.Println(http.ListenAndServe(":8080", nil))
}
```

---

## 【文本块 79】Q：重试为什么有风险？

重试看似能提高成功率，但也可能放大故障。

如果下游已经很慢或快挂了，上游继续重试，相当于把流量乘以重试次数打过去，可能导致雪崩。

重试必须满足几个条件：

1. 有明确超时
2. 限制重试次数
3. 使用指数退避和 jitter
4. 只对可重试错误重试
5. 写请求必须幂等
6. 配合熔断和限流

比如 GET 请求通常更适合重试。
POST 扣款、下单这类非幂等请求，不能盲目重试。

面试回答：

重试不是越多越稳。没有超时和幂等的重试会放大故障，尤其是写请求可能导致重复下单、重复扣款。生产里重试要配合超时、次数限制、退避、幂等和熔断。

---

## 【文本块 80】Q：HTTP 连接复用为什么重要？

HTTP client 如果每次请求都新建 TCP 连接，会产生：

1. TCP 握手开销
2. TLS 握手开销
3. TIME_WAIT 增多
4. 延迟增加
5. CPU 增加
6. 端口资源消耗

Go 的 http.Transport 默认支持连接复用，但前提是：

1. response body 被读完或丢弃
2. response body 被关闭
3. Transport 没有被频繁新建
4. 连接没有被服务端关闭

所以生产里不要每次请求都创建新的 http.Client 和 Transport。

应该复用一个全局 client，并合理配置连接池。

---

# 二十六、本章速记版

## 【文本块 81】操作系统速记

进程是资源分配单位，线程是 CPU 调度单位，协程是用户态轻量调度单位。

goroutine 是 Go runtime 管理的轻量执行单元，最终运行在 OS 线程上。

用户态权限低，内核态权限高。文件 IO、网络 IO、进程管理通常要通过系统调用进入内核态。

系统调用是应用主动请求内核服务，中断来自外部设备或时钟，异常来自当前程序执行中的特殊事件。

上下文切换贵在保存恢复执行状态、缓存局部性变差、进程切换可能导致 TLB 失效。

虚拟内存提供进程隔离、地址空间抽象和内存扩展。

页表负责虚拟页到物理页框映射，TLB 是页表缓存。

缺页中断不一定是错误，但频繁缺页和 swap 会严重拖慢服务。

死锁四条件：互斥、请求并保持、不可剥夺、循环等待。

避免死锁最常用：固定加锁顺序、缩短持锁时间、减少嵌套锁。

IO 多路复用让一个线程监听多个 fd，epoll 适合大量连接少量活跃事件。

Reactor 是事件分发模型，epoll 是事件发现机制。

零拷贝减少用户态和内核态之间的数据复制，IO 多路复用减少线程数量。

CPU 缓存依赖局部性，缓存行是传输单位，伪共享是多个变量落在同一缓存行导致的性能问题。

---

## 【文本块 82】Linux 排障速记

CPU：

```bash
top
top -Hp <pid>
ps -ef
```

内存：

```bash
free -m
vmstat
pmap -x <pid>
```

磁盘：

```bash
df -h
du -sh *
iostat -x 1
```

网络：

```bash
ss -lntp
ss -antp
lsof -i :8080
curl -v
tcpdump
```

日志：

```bash
tail -f
grep
less
journalctl
```

Go：

```bash
go tool pprof
go tool trace
/debug/pprof/goroutine?debug=2
/debug/pprof/heap
/debug/pprof/profile
```

排障顺序：

先看资源，再定位进程和端口，再看日志和链路，最后看代码和依赖。

---

## 【文本块 83】网络速记

网络分层是为了职责隔离和降低复杂度。

TCP/IP 四层：

应用层：HTTP、DNS、gRPC。
传输层：TCP、UDP。
网络层：IP、ICMP。
网络接口层：以太网、ARP。

TCP 面向连接、可靠、有序、字节流，有流控和拥塞控制。
UDP 无连接、不保证可靠和顺序，延迟低，适合实时场景。

三次握手：确认双方收发能力，交换初始序列号。
四次挥手：TCP 全双工，两个方向分别关闭。

TIME_WAIT 在主动关闭方，作用是可靠关闭和避免旧报文影响新连接。
CLOSE_WAIT 在被动关闭方，通常说明应用没正确 close。

TCP 是字节流协议，没有消息边界，所以会有粘包拆包。
常见解决：固定长度、分隔符、长度字段。

HTTP/1.1 默认长连接。
HTTP/2 二进制分帧、多路复用、头部压缩。
HTTP/3 基于 QUIC/UDP，改善 TCP 层队头阻塞。

HTTPS = HTTP + TLS，提供加密、身份认证和完整性保护。

DNS 解析有多级缓存。
CDN 通过 DNS 调度和边缘缓存降低延迟和源站压力。

网络排查按 DNS、连通性、端口、协议、应用逐层定位。

---

# 二十七、本章最终面试回答模板

## 【文本块 84】操作系统 + 网络综合回答模板

如果面试官让我系统讲操作系统和网络，我会先把它们和 Go 后端联系起来。

操作系统这边，我会先讲进程、线程和 goroutine 的关系。进程是资源隔离单位，线程是 CPU 调度单位，而 goroutine 是 Go runtime 管理的轻量级执行单元。Go 通过 GMP 调度模型把大量 goroutine 复用到较少的 OS 线程上执行，所以我们可以用 goroutine-per-request 或 goroutine-per-connection 的方式写高并发代码。但 goroutine 不是免费的，如果 channel、context、网络 IO 处理不好，也会出现 goroutine 泄漏。

然后我会讲用户态和内核态。Go 程序运行在用户态，读写文件、网络通信、创建线程、申请某些资源时，需要通过系统调用进入内核态。系统调用是主动请求内核服务，中断通常来自外部设备或时钟，异常来自程序执行中的特殊事件。上下文切换的成本不只是保存寄存器，还包括缓存局部性变差、进程切换可能导致 TLB 失效，所以高性能系统要避免线程过多和频繁切换。

内存方面，我会讲虚拟内存、分页、页表和 TLB。虚拟内存给每个进程提供独立连续的地址空间，操作系统通过页表把虚拟页映射到物理页框，TLB 用来缓存热点地址映射。缺页中断不一定是错误，但如果内存压力大导致频繁缺页甚至 swap，Go 服务延迟会明显抖动。Go 里还要结合逃逸分析、GC、heap profile 来看内存问题。

IO 方面，我会讲 select、poll、epoll。IO 多路复用的核心是一个线程监听多个 fd，就绪了再处理。select 有数量限制和全量遍历问题，poll 解决了数量限制但仍然要遍历，epoll 基于事件驱动，更适合大量连接少量活跃事件。Go 网络代码看起来是阻塞式 Read/Write，但 runtime 底层通过非阻塞 fd 和 netpoller 挂起等待 IO 的 goroutine，因此不会一个连接占死一个 OS 线程。

网络方面，我会从分层开始讲。网络分层是为了职责隔离，应用层关心业务数据，传输层关心端到端通信，网络层关心路由寻址，链路层关心局域网内帧转发。TCP 和 UDP 的核心区别是：TCP 面向连接、可靠、有序、字节流，有流控和拥塞控制；UDP 无连接、不保证可靠和顺序，但延迟低。HTTP、MySQL、Redis、RPC 通常跑在 TCP 上，DNS、音视频、实时游戏等常见 UDP 场景更多。

TCP 这块，我会讲三次握手和四次挥手。三次握手是为了确认双方收发能力并同步初始序列号，第三次握手让服务端确认客户端能收到自己的 SYN+ACK。四次挥手是因为 TCP 是全双工协议，两个方向要分别关闭。TIME_WAIT 出现在主动关闭方，用于可靠关闭和避免旧报文影响新连接；CLOSE_WAIT 出现在被动关闭方，如果很多，通常说明应用没有正确 close 连接。Go 里常见原因是 HTTP response body 没有关闭。

HTTP 这块，我会讲 HTTP/1.1、HTTP/2、HTTP/3 和 HTTPS。HTTP/1.1 默认长连接，但复用能力有限；HTTP/2 通过二进制分帧和多路复用提升效率，但仍受 TCP 丢包影响；HTTP/3 基于 QUIC/UDP，改善 TCP 层队头阻塞。HTTPS 本质是 HTTP 加 TLS，TLS 握手用非对称加密和证书完成身份认证和密钥协商，后续数据传输用对称加密提高性能。

工程实践上，Go HTTP server 要配置 ReadHeaderTimeout、ReadTimeout、WriteTimeout 和 IdleTimeout，防止慢连接拖垮服务。Go HTTP client 要配置 Timeout、Transport 连接池、Dial 超时、TLS 握手超时、响应头超时，并且必须关闭 resp.Body，才能复用连接。所有下游调用都应该传 context，避免慢依赖无限占用 goroutine、连接和内存。重试要谨慎，必须配合超时、次数限制、退避和幂等，否则会放大故障。

线上排查时，我会按链路分层。CPU 用 top 和 pprof，内存用 free、vmstat、heap profile，磁盘用 df、du、iostat，网络用 ss、lsof、curl、tcpdump，DNS 用 dig/nslookup，路径问题用 traceroute。排查网络问题时，按 DNS、连通性、端口、协议、应用日志逐层缩小范围。Go 服务则重点看 pprof、goroutine dump、连接池指标、context timeout、错误日志和 trace 链路。

一句话总结：

操作系统决定 Go 服务如何使用 CPU、内存、线程、文件和网络；计算机网络决定 Go 服务如何和外部系统通信。Go 后端不一定要手写 epoll 或 TCP 协议栈，但必须理解 runtime netpoller、context、net/http、连接池、pprof 和 Linux 排障背后的系统原理。
